# Orbit Surface Revolution

This example builds an explicit ellipse path, revolves it into a surface, samples that surface, and then runs the pressure and torque diagnostics on the sampled surface.


Trajectory note: this notebook provides explicit `time` and trajectory `velocity_xyz` when sampling.
The library functions consume those directly; no periodic-orbit reconstruction is used in the library.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from starwinds_analysis.analysis.orbits import elliptic_orbit_points
from starwinds_analysis.analysis.orbits import trajectory_velocity
from starwinds_analysis.physics.orbits import orbital_period
from starwinds_analysis.physics.orbit_surface import pressure_components_on_surface
from starwinds_analysis.physics.orbit_surface import sample_surface_revolution
from starwinds_analysis.physics.orbit_surface import surface_of_revolution_from_trajectory
from starwinds_analysis.physics.orbit_surface import torque_components_on_surface
from starwinds_analysis.smart_ds import SmartDs


In [ ]:
data_file = Path('../sample_data/3d__var_4_n00000000.plt')
sds = SmartDs.from_file(str(data_file))
sds.prepare()

path = elliptic_orbit_points(10.0, eccentricity=0.2, n_points=64, return_info=True)
surface = surface_of_revolution_from_trajectory(path['points'], n_longitudes=48)

print('surface points:', surface['points'].shape)


In [ ]:
body_radius = float(sds('star_radius [m]'))
star_mass = float(sds('star_mass [kg]'))
period = orbital_period(10.0 * body_radius, star_mass)
time = path['phase [turns]'] * period
velocity = trajectory_velocity(
    path['points'],
    time,
    coordinate_scale=body_radius,
)

sampled = sample_surface_revolution(
    sds,
    fields=(
        'mass_flux [kg/m^2/s]',
        'Rho [kg/m^3]',
        'U_x [m/s]',
        'U_y [m/s]',
        'U_z [m/s]',
        'B_x [T]',
        'B_y [T]',
        'B_z [T]',
        'U [m/s]',
        'B [T]',
        'magnetic_pressure [Pa]',
        'ram_pressure [Pa]',
        'thermal_pressure [Pa]',
        'standoff_distance [m]',
    ),
    trajectory_points=path['points'],
    phase=path['phase [turns]'],
    time=time,
    time_weight=path['time_weight [none]'],
    velocity_xyz=velocity,
    trajectory_meta={
        'semi_major_axis [R]': 10.0,
        'eccentricity [none]': 0.2,
    },
    n_longitudes=48,
    method='nearest',
)

print('sampled mass-flux shape:', sampled('mass_flux [kg/m^2/s]').shape)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
img = ax.imshow(
    sampled('mass_flux [kg/m^2/s]'),
    aspect='auto',
    origin='lower',
    extent=(0.0, 2.0 * np.pi, 0.0, 1.0),
    cmap='viridis',
)
ax.set_xlabel('azimuth [rad]')
ax.set_ylabel('phase [turns]')
ax.set_title('mass_flux [kg/m^2/s] on revolved surface')
fig.colorbar(img, ax=ax, label='mass_flux [kg/m^2/s]')
plt.show()


In [ ]:
pressure = pressure_components_on_surface(
    sampled,
)

print('finite relative ram:', np.count_nonzero(np.isfinite(pressure['relative_ram_pressure [Pa]'])))
print('phase bins:', pressure['phase_quantiles']['ram_pressure [Pa]']['values'].shape)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
img = ax.imshow(
    pressure['relative_ram_pressure [Pa]'],
    aspect='auto',
    origin='lower',
    extent=(0.0, 2.0 * np.pi, 0.0, 1.0),
    cmap='magma',
)
ax.set_xlabel('azimuth [rad]')
ax.set_ylabel('phase [turns]')
ax.set_title('relative_ram_pressure [Pa] on revolved surface')
fig.colorbar(img, ax=ax, label='relative_ram_pressure [Pa]')
plt.show()


In [ ]:
torque = torque_components_on_surface(
    sampled,
    body_radius=body_radius,
    angvel=float(sds('star_rotation_rate [rad/s]')),
)

print('T1_magnetic [Nm]:', float(torque['T1_magnetic [Nm]']))
print('T4_dynamic [Nm]:', float(torque['T4_dynamic [Nm]']))
print('total [Nm]:', float(torque['total [Nm]']))
print('coverage [none]:', float(torque['coverage [none]']))
